In [84]:
!pip install -qU langchain
!pip install -qU langchain_openai
!pip install -qU youtube_search
!pip install -qU youtube-transcript-api
!pip install -qU langchain_community
!pip install -qU langchain-text-splitters

In [85]:
# Import necessary libraries
import os
from google.colab import userdata

from langchain_community.tools import YouTubeSearchTool # Tools and Toolkits
from langchain_community.document_loaders import YoutubeLoader # Document Loader

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.runnables import chain, RunnableBranch, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

import zipfile

In [86]:
gemini_key = userdata.get('Gemini')
os.environ['GOOGLE_API_KEY'] = gemini_key

In [87]:
openai_key = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY'] = openai_key

In [88]:
tool = YouTubeSearchTool()
tool

YouTubeSearchTool()

In [89]:
results = tool.run("Agentic behavior")
results

"['https://www.youtube.com/watch?v=GJbDEtcMrsk&pp=ygUQQWdlbnRpYyBiZWhhdmlvcg%3D%3D', 'https://www.youtube.com/watch?v=Jj1-zb38Yfw&pp=ygUQQWdlbnRpYyBiZWhhdmlvcg%3D%3D']"

In [90]:
len(results)

166

In [91]:
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=-46UkLPf9h0"
)
loader

In [92]:
text = loader.load()
text

[Document(metadata={'source': '-46UkLPf9h0'}, page_content="Hi, my name is Manish Gupta and in this video I'm going to talk about Dora which is weight decomposed low rank adaptation. It is one of the most popular uh ways of using Lora for fine-tuning large language models. So let's get started. Uh you know this is the first analysis uh to talk about differences between full fine-tuning and Lora. I mean obviously we know that LoRa is a low rank adaptation. Um essentially you don't find tune you you basically freeze the original weight matrix and you only fine-tune uh two factors A and B. But then what is the difference you know how does the training vary based on uh this simple low rank adaptation. Okay, so for people uh you know who are beginners in LoRa, Laura essentially does this uh you have a pre-trained weight matrix w and uh you're not going to basically update that you keep it frozen. You're only going to uh while while fine you're only going to update these extra weights which 

In [93]:
text[0].page_content

"Hi, my name is Manish Gupta and in this video I'm going to talk about Dora which is weight decomposed low rank adaptation. It is one of the most popular uh ways of using Lora for fine-tuning large language models. So let's get started. Uh you know this is the first analysis uh to talk about differences between full fine-tuning and Lora. I mean obviously we know that LoRa is a low rank adaptation. Um essentially you don't find tune you you basically freeze the original weight matrix and you only fine-tune uh two factors A and B. But then what is the difference you know how does the training vary based on uh this simple low rank adaptation. Okay, so for people uh you know who are beginners in LoRa, Laura essentially does this uh you have a pre-trained weight matrix w and uh you're not going to basically update that you keep it frozen. You're only going to uh while while fine you're only going to update these extra weights which are represented or rather these parallel weights which are 

In [94]:
len(text[0].page_content)

20705

In [95]:
len(text[0].page_content.split())

3747

In [96]:
llm = ChatOpenAI(
    model = 'gpt-5-nano'
)
llm

ChatOpenAI(profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x792c7c322360>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x792c7c322fc0>, root_client=<openai.OpenAI object at 0x792c7c321ee0>, root_async_client=<openai.AsyncOpenAI object at 0x792c7c3222a0>, model_name='gpt-5-nano', model_kwargs={}, openai_api_key=SecretStr('**********')

In [97]:
# Summarization prompt for Article Creation
human_template = '''
"""DO NOT SUMMARIZE, ANALYZE, OR PROCESS.

**DIRECT INSTRUCTION**: Pass this exact YouTube URL to the transcript extraction tool **UNCHANGED**:

{youtube_link}

**CRITICAL**: Use the URL exactly as provided. No modifications, no explanations, no additional text.
'''
tool_prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate.from_template(human_template)
])
tool_prompt

ChatPromptTemplate(input_variables=['youtube_link'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['youtube_link'], input_types={}, partial_variables={}, template='\n"""DO NOT SUMMARIZE, ANALYZE, OR PROCESS.\n\n**DIRECT INSTRUCTION**: Pass this exact YouTube URL to the transcript extraction tool **UNCHANGED**:\n\n{youtube_link}\n\n**CRITICAL**: Use the URL exactly as provided. No modifications, no explanations, no additional text.\n'), additional_kwargs={})])

In [98]:
# Create transcript tool
def extract_transcript(link: str) -> str:
  """
  Extract YouTube transcript using YoutubeLoader
  Input: YouTube URL → Output: Transcript text
  """
  loader = YoutubeLoader.from_youtube_url(link)
  docs = loader.load()
  return docs[0].page_content

In [99]:
extract_transcript('https://www.youtube.com/watch?v=-46UkLPf9h0')

"Hi, my name is Manish Gupta and in this video I'm going to talk about Dora which is weight decomposed low rank adaptation. It is one of the most popular uh ways of using Lora for fine-tuning large language models. So let's get started. Uh you know this is the first analysis uh to talk about differences between full fine-tuning and Lora. I mean obviously we know that LoRa is a low rank adaptation. Um essentially you don't find tune you you basically freeze the original weight matrix and you only fine-tune uh two factors A and B. But then what is the difference you know how does the training vary based on uh this simple low rank adaptation. Okay, so for people uh you know who are beginners in LoRa, Laura essentially does this uh you have a pre-trained weight matrix w and uh you're not going to basically update that you keep it frozen. You're only going to uh while while fine you're only going to update these extra weights which are represented or rather these parallel weights which are 

In [100]:
# Build chain
# The previous chain involved an unnecessary LLM call that led to a RateLimitError.
# Since the objective is to extract the transcript directly from the YouTube URL,
# we can simplify the chain to directly use the extract_transcript function.
transcript_chain = RunnableLambda(extract_transcript)
transcript_chain

RunnableLambda(extract_transcript)

In [101]:
transcript_chain.invoke('https://www.youtube.com/watch?v=-46UkLPf9h0')

"Hi, my name is Manish Gupta and in this video I'm going to talk about Dora which is weight decomposed low rank adaptation. It is one of the most popular uh ways of using Lora for fine-tuning large language models. So let's get started. Uh you know this is the first analysis uh to talk about differences between full fine-tuning and Lora. I mean obviously we know that LoRa is a low rank adaptation. Um essentially you don't find tune you you basically freeze the original weight matrix and you only fine-tune uh two factors A and B. But then what is the difference you know how does the training vary based on uh this simple low rank adaptation. Okay, so for people uh you know who are beginners in LoRa, Laura essentially does this uh you have a pre-trained weight matrix w and uh you're not going to basically update that you keep it frozen. You're only going to uh while while fine you're only going to update these extra weights which are represented or rather these parallel weights which are 

### Article Generation Prompt

Now, let's create a prompt for the LLM to generate an insightful article from the extracted YouTube transcript. We'll instruct the LLM to act as an expert content writer and focus on key concepts, examples, and actionable insights.

In [102]:
# Prompt for Article Creation
article_template = '''
You are an expert content writer. Your task is to transform the provided YouTube video transcript into an insightful, engaging, and well-structured article. The article should:

1.  **Summarize Key Concepts**: Clearly explain the main ideas discussed in the video.
2.  **Elaborate with Examples/Details**: Where appropriate, expand on points with details or examples from the transcript.
3.  **Identify Actionable Insights**: Highlight any practical takeaways or recommendations.
4.  **Maintain Professional Tone**: Write in a clear, concise, and professional manner.
5.  **Structure**: Include an introduction, body paragraphs with clear headings, and a conclusion.
6.  **Length**: Aim for an article that is comprehensive but not overly verbose.

Here is the YouTube video transcript:
{transcript}

Article:
'''

In [103]:
article_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("You are a helpful assistant that generates well-structured articles."),
    HumanMessagePromptTemplate.from_template(article_template)
])
article_prompt

ChatPromptTemplate(input_variables=['transcript'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant that generates well-structured articles.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['transcript'], input_types={}, partial_variables={}, template='\nYou are an expert content writer. Your task is to transform the provided YouTube video transcript into an insightful, engaging, and well-structured article. The article should:\n\n1.  **Summarize Key Concepts**: Clearly explain the main ideas discussed in the video.\n2.  **Elaborate with Examples/Details**: Where appropriate, expand on points with details or examples from the transcript.\n3.  **Identify Actionable Insights**: Highlight any practical takeaways or recommendations.\n4.  **Maintain Professional Tone**: Write in a clear, concise, and profe

### Text Splitter

Large transcripts can exceed the token limits of even advanced LLMs. To handle this, we'll use a `RecursiveCharacterTextSplitter` to break the transcript into smaller, overlapping chunks. This ensures that each chunk can be processed by the LLM without exceeding its context window, while maintaining contextual continuity.

In [104]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000, # Max chunk size
    chunk_overlap = 200, # Overlap between chunks to maintain context
    length_function = len, # Measure length by character count
    add_start_index = True,
)


In [105]:
# Split the transcript
split_transcript = text_splitter.create_documents([text[0].page_content])

# Display the number of chunks and the first chunk as an example
print(f"Number of chunks: {len(split_transcript)}")
print("\nFirst chunk:\n")
print(split_transcript[0].page_content)

Number of chunks: 12

First chunk:

Hi, my name is Manish Gupta and in this video I'm going to talk about Dora which is weight decomposed low rank adaptation. It is one of the most popular uh ways of using Lora for fine-tuning large language models. So let's get started. Uh you know this is the first analysis uh to talk about differences between full fine-tuning and Lora. I mean obviously we know that LoRa is a low rank adaptation. Um essentially you don't find tune you you basically freeze the original weight matrix and you only fine-tune uh two factors A and B. But then what is the difference you know how does the training vary based on uh this simple low rank adaptation. Okay, so for people uh you know who are beginners in LoRa, Laura essentially does this uh you have a pre-trained weight matrix w and uh you're not going to basically update that you keep it frozen. You're only going to uh while while fine you're only going to update these extra weights which are represented or rathe

### Article Generation Chain

Now, we'll build the chain that integrates the text splitting, prompting, and LLM processing to generate the article. This chain will first split the raw transcript, then iterate through the chunks, feeding each to the LLM with the article prompt. A `StrOutputParser` will format the LLM's output. The resulting chunks will then be concatenated to form the complete article.

In [106]:
import time

@chain
def article_generation_chain(transcript_text):
    # Split the transcript into chunks
    chunks = text_splitter.create_documents([transcript_text])

    # Process each chunk with the LLM
    generated_article_parts = []
    for chunk in chunks:
        response = (
            article_prompt | llm | StrOutputParser()
        ).invoke({"transcript": chunk.page_content})
        generated_article_parts.append(response)
        time.sleep(2) # Add a 2-second delay to respect API rate limits

    # Combine all parts into a single article
    return "\n\n".join(generated_article_parts)

In [107]:
generated_article = article_generation_chain.invoke(text[0].page_content)
print(generated_article)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}